<a href="https://colab.research.google.com/github/vuongtran31251023271-code/Th-/blob/main/App_giao_h%C3%A0ng_HTV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit scikit-fuzzy geopy folium streamlit-folium
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 523.7/523.7 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 4s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from geopy.distance import geodesic
import folium
from streamlit_folium import st_folium
import time
import requests

st.set_page_config(page_title="APP GIAO HÀNG NHANH", layout="wide")

def get_driving_route(c1, c2):
    try:
        url = f"http://router.project-osrm.org/route/v1/driving/{c1[1]},{c1[0]};{c2[1]},{c2[0]}?overview=full&geometries=geojson"
        res = requests.get(url).json()
        route = res['routes'][0]
        coords = route['geometry']['coordinates']
        route_coords = [[lat, lon] for lon, lat in coords]
        dist_km = round(route['distance'] / 1000.0, 2)
        base_time = round(route['duration'] / 60.0, 1)
        return route_coords, dist_km, base_time
    except:
        dist_km = round(geodesic(c1, c2).km * 1.3, 2)
        return [c1, c2], dist_km, dist_km * 3

if 'order_step' not in st.session_state:
    st.session_state.order_step = 1
    st.session_state.dist_km = 0
    st.session_state.base_time = 0
    st.session_state.base_price = 0
    st.session_state.coords_1 = None
    st.session_state.coords_2 = None
    st.session_state.route_coords = []

@st.cache_resource
def setup_fuzzy():
    f_dist = ctrl.Antecedent(np.arange(0, 21, 1), 'distance')
    f_traffic = ctrl.Antecedent(np.arange(0, 11, 1), 'traffic')
    f_weather = ctrl.Antecedent(np.arange(0, 11, 1), 'weather')
    f_fatigue = ctrl.Antecedent(np.arange(0, 11, 1), 'fatigue')
    f_prep = ctrl.Antecedent(np.arange(0, 31, 1), 'prep_time')

    f_time = ctrl.Consequent(np.arange(0, 61, 1), 'est_time')
    f_bonus = ctrl.Consequent(np.arange(0, 101, 1), 'bonus')
    f_rating = ctrl.Consequent(np.arange(1, 6, 1), 'rating')

    f_dist['S'] = fuzz.trimf(f_dist.universe, [0, 0, 3])
    f_dist['M'] = fuzz.trimf(f_dist.universe, [3, 5, 8])
    f_dist['L'] = fuzz.trapmf(f_dist.universe, [8, 12, 20, 20])

    f_traffic['L'] = fuzz.trimf(f_traffic.universe, [0, 0, 4])
    f_traffic['M'] = fuzz.trimf(f_traffic.universe, [3, 5, 7])
    f_traffic['H'] = fuzz.trimf(f_traffic.universe, [6, 10, 10])

    f_weather['C'] = fuzz.trimf(f_weather.universe, [0, 0, 4])
    f_weather['R'] = fuzz.trimf(f_weather.universe, [3, 5, 7])
    f_weather['S'] = fuzz.trimf(f_weather.universe, [6, 10, 10])

    f_fatigue['L'] = fuzz.trimf(f_fatigue.universe, [0, 0, 4])
    f_fatigue['M'] = fuzz.trimf(f_fatigue.universe, [3, 5, 7])
    f_fatigue['H'] = fuzz.trimf(f_fatigue.universe, [6, 10, 10])

    f_prep['F'] = fuzz.trimf(f_prep.universe, [0, 0, 5])
    f_prep['M'] = fuzz.trimf(f_prep.universe, [5, 10, 15])
    f_prep['S'] = fuzz.trapmf(f_prep.universe, [15, 20, 30, 30])

    f_time['S'] = fuzz.trimf(f_time.universe, [0, 0, 10])
    f_time['M'] = fuzz.trimf(f_time.universe, [10, 17, 25])
    f_time['L'] = fuzz.trapmf(f_time.universe, [25, 30, 60, 60])

    f_bonus['L'] = fuzz.trimf(f_bonus.universe, [0, 0, 15])
    f_bonus['M'] = fuzz.trimf(f_bonus.universe, [10, 25, 40])
    f_bonus['H'] = fuzz.trapmf(f_bonus.universe, [35, 60, 100, 100])

    f_rating['P'] = fuzz.trimf(f_rating.universe, [1, 1, 2])
    f_rating['A'] = fuzz.trimf(f_rating.universe, [2, 3, 4])
    f_rating['E'] = fuzz.trapmf(f_rating.universe, [4, 5, 5, 5])

    rule1 = ctrl.Rule(f_traffic['L'] & f_dist['S'], f_time['S'])
    rule2 = ctrl.Rule(f_traffic['M'] & f_dist['M'], f_time['M'])
    rule3 = ctrl.Rule(f_traffic['H'] & f_dist['L'], f_time['L'])
    rule4 = ctrl.Rule(f_weather['C'], f_bonus['L'])
    rule5 = ctrl.Rule(f_weather['R'], f_bonus['M'])
    rule6 = ctrl.Rule(f_weather['S'], f_bonus['H'])
    rule7 = ctrl.Rule(f_prep['F'] & f_traffic['L'], f_time['S'])
    rule8 = ctrl.Rule(f_prep['M'] & f_traffic['M'], f_time['M'])
    rule9 = ctrl.Rule(f_prep['S'] & f_traffic['H'], f_time['L'])
    rule10 = ctrl.Rule(f_fatigue['L'], f_rating['E'])
    rule11 = ctrl.Rule(f_fatigue['M'], f_rating['A'])
    rule12 = ctrl.Rule(f_fatigue['H'], f_rating['P'])
    rule13 = ctrl.Rule(f_dist['L'] & f_weather['S'] & f_traffic['H'], (f_time['L'], f_bonus['H']))
    rule14 = ctrl.Rule(f_dist['S'] & f_weather['C'] & f_traffic['L'], (f_time['S'], f_bonus['L']))
    rule15 = ctrl.Rule(f_fatigue['H'] & f_traffic['H'], (f_rating['P'], f_time['L']))

    fuzzy_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10, rule11, rule12, rule13, rule14, rule15])
    return ctrl.ControlSystemSimulation(fuzzy_ctrl)

fuzzy_sim = setup_fuzzy()

LOCATIONS = {
    "Chợ Bến Thành, Quận 1": (10.7725, 106.6980),
    "Landmark 81, Bình Thạnh": (10.7946, 106.7218),
    "Phố đi bộ Nguyễn Huệ, Quận 1": (10.7743, 106.7031),
    "Sân bay Tân Sơn Nhất, Tân Bình": (10.8149, 106.6634),
    "Đại học Bách Khoa, Quận 10": (10.7733, 106.6596),
    "Aeon Mall Tân Phú": (10.8005, 106.6166)
}

if st.button("RESET APP"):
    st.session_state.clear()
    st.rerun()

col1, col2 = st.columns([1, 2])

with col1:
    st.subheader("1. Thông Tin Đơn Hàng")
    is_locked = (st.session_state.order_step > 1)

    pickup_choice = st.selectbox("Chọn điểm LẤY HÀNG:", list(LOCATIONS.keys()), index=1, disabled=is_locked)
    dropoff_choice = st.selectbox("Chọn điểm GIAO HÀNG:", list(LOCATIONS.keys()), index=4, disabled=is_locked)

    if st.session_state.order_step == 1:
        if st.button("🔍 TÌM TÀI XẾ & VẼ ĐƯỜNG", type="primary", use_container_width=True):
            with st.spinner("Đang dò đường ngắn nhất..."):
                c1 = LOCATIONS[pickup_choice]
                c2 = LOCATIONS[dropoff_choice]

                route_coords, dist_km, base_time = get_driving_route(c1, c2)

                st.session_state.route_coords = route_coords
                st.session_state.dist_km = dist_km
                st.session_state.base_time = base_time
                st.session_state.base_price = int(dist_km * 15000)
                st.session_state.coords_1 = c1
                st.session_state.coords_2 = c2
                st.session_state.order_step = 2
                st.rerun()

    if st.session_state.order_step >= 2:
        st.success("✅ TÀI XẾ ĐÃ NHẬN ĐƠN!")
        st.divider()
        st.subheader("2. Tình Trạng Thực Tế ")

        is_done = (st.session_state.order_step == 3)
        distance_val = st.slider("Khoảng cách đến khách hàng (km)", 0, 20, int(st.session_state.dist_km), disabled=is_done)
        traffic_val = st.slider("Mức độ kẹt xe (0-10)", 0, 10, 5, disabled=is_done)
        weather_val = st.slider("Thời tiết (0: Đẹp - 10: Bão)", 0, 10, 0, disabled=is_done)
        prep_val = st.slider("TG chuẩn bị món (phút)", 0, 30, 10, disabled=is_done)
        fatigue_val = st.slider("Độ mệt mỏi của tài xế (0-10)", 0, 10, 2, disabled=is_done)

with col2:
    if st.session_state.order_step >= 2:
        st.subheader("📍 Bản đồ định tuyến ")
        m = folium.Map(location=[(st.session_state.coords_1[0] + st.session_state.coords_2[0])/2,
                                 (st.session_state.coords_1[1] + st.session_state.coords_2[1])/2], zoom_start=13)

        folium.PolyLine(st.session_state.route_coords, color="blue", weight=5, opacity=0.7).add_to(m)

        folium.Marker(st.session_state.coords_1, popup="Lấy Hàng", icon=folium.Icon(color="blue", icon="info-sign")).add_to(m)
        folium.Marker(st.session_state.coords_2, popup="Giao Hàng", icon=folium.Icon(color="green", icon="ok-sign")).add_to(m)
        st_folium(m, width=700, height=350)

        fuzzy_sim.input['distance'] = distance_val
        fuzzy_sim.input['traffic'] = traffic_val
        fuzzy_sim.input['weather'] = weather_val
        fuzzy_sim.input['prep_time'] = prep_val
        fuzzy_sim.input['fatigue'] = fatigue_val


        try:
            fuzzy_sim.compute()
            est_time = int(fuzzy_sim.output.get('est_time', st.session_state.base_time + prep_val))
            bonus = int(fuzzy_sim.output.get('bonus', weather_val * 4))
            rating = round(fuzzy_sim.output.get('rating', 4.0), 1)
        except:
            est_time = int(st.session_state.base_time + prep_val + (traffic_val * 1.5))
            bonus = int(weather_val * 5)
            rating = round(max(1.0, 5.0 - fatigue_val*0.3), 1)

        bonus_amount = int(st.session_state.base_price * (bonus / 100))

        m1, m2, m3 = st.columns(3)
        m1.metric("Quãng Đường Thực Tế", f"{st.session_state.dist_km} km")
        m2.metric("Thời Gian Dự Kiến", f"{est_time} phút")
        m3.metric("Tổng Cước (+Thưởng)", f"{st.session_state.base_price + bonus_amount:,} VNĐ")

        st.divider()

        if st.session_state.order_step == 2:
            if st.button("📦 MÔ PHỎNG: ĐÃ GIAO HÀNG XONG", type="primary", use_container_width=True):
                st.session_state.final_rating = rating
                st.session_state.order_step = 3
                st.rerun()

        if st.session_state.order_step == 3:
            st.success("🎉 Đơn hàng đã hoàn thành!")
            st.subheader("🌟 Đánh Giá Tài Xế (Từ Hệ Thống Mờ)")
            st.metric("Điểm Tài Xế Nhận Được:", f"{st.session_state.final_rating} / 5 ⭐")

            if st.button("🔄 Bắt đầu cuốc xe mới", use_container_width=True):
                st.session_state.order_step = 1
                st.rerun()

Writing app.py


In [ ]:

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


!pkill -f streamlit
!pkill -f localtunnel

import subprocess
import time
subprocess.Popen(["streamlit", "run", "app.py"])
print("Đang khởi động web, vui lòng đợi 3 giây...")
time.sleep(3)

print("ĐANG TẠO LINK CLOUDFLARE...")
!./cloudflared tunnel --url http://localhost:8501

--2026-03-10 14:17:05--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-03-10 14:17:05--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-10T15%3A07%3A57Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-10T1